In [7]:
import os, re, io, shutil, time, requests, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
import plotly.graph_objects as go
import statsmodels.api as sm
from reportlab.lib.pagesizes import letter, landscape
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, Image
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
import warnings, traceback
warnings.filterwarnings('ignore', message='Data Validation extension is not supported and will be removed', category=UserWarning, module='openpyxl.worksheet._reader')
pd.options.display.float_format = '{:,.2f}'.format
DEFAULT_PATH = os.path.join(os.getcwd(), 'PlantillaBC_2Grupo No. 1.xlsx')
OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

def _to_number(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float, np.number)): return float(x)
    s = str(x).strip().replace('\xa0',' ').replace('%','')
    s = s.replace(' ', '')
    if s.count(',') > 1 and '.' in s:
        s = s.replace(',', '')
    elif s.count('.') > 1 and ',' in s:
        s = s.replace('.', '').replace(',', '.')
    else:
        if ',' in s and '.' not in s:
            s = s.replace(',', '.')
        s = s.replace(',', '')
    try: return float(s)
    except: return np.nan

def parse_multi_year_sheet(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name, header=None)
    hdr = None
    for r in range(min(60, df.shape[0])):
        if str(df.iat[r, 0]).strip().upper() == 'CONCEPTOS':
            hdr = r; break
    if hdr is None:
        raise ValueError(f"No se encontró 'CONCEPTOS' en la hoja {sheet_name}")
    date_row = hdr + 1
    def _get_year_from(cell):
        dt = pd.to_datetime(cell, errors='coerce')
        if pd.isna(dt): return None
        return int(dt.year)
    amount_cols, year_labels = [], []
    for c in range(1, df.shape[1]):
        top = df.iat[hdr, c]
        below = df.iat[date_row, c] if date_row < df.shape[0] else None
        y = None
        if isinstance(top, str) and top.strip().upper() == 'FECHA':
            y = _get_year_from(below)
        else:
            y = _get_year_from(top)
        if y is not None:
            amount_cols.append(c)
            year_labels.append(str(y))
    if not amount_cols:
        raise ValueError(f'No se detectaron columnas de año en {sheet_name}')
    start = hdr + 2
    sub = df.iloc[start:, [0] + amount_cols].copy()
    temp_cols = ['Cuenta'] + [f"Y{i}_{y}" for i, y in enumerate(year_labels)]
    sub.columns = temp_cols
    sub['Cuenta'] = sub['Cuenta'].astype(str).str.strip()
    for c in temp_cols[1:]: sub[c] = sub[c].apply(_to_number)
    uniq_years = sorted(set(year_labels))
    out = pd.DataFrame({'Cuenta': sub['Cuenta']})
    for y in uniq_years:
        cols_y = [c for c in temp_cols[1:] if c.endswith(f'_{y}')]
        out[y] = sub[cols_y].sum(axis=1, skipna=True)
    out = out.dropna(how='all', subset=uniq_years).reset_index(drop=True)
    return out, uniq_years

def vertical_analysis_exact(df, year, total_patterns, prefer_max=True):
    out = df[['Cuenta', year]].copy()
    mask = False
    for pat in total_patterns:
        mask = mask | out['Cuenta'].str.contains(pat, case=False, regex=True, na=False)
    matches = out.loc[mask, year]
    if not matches.empty:
        total = (matches.max(skipna=True) if prefer_max else matches.sum(skipna=True))
    else:
        total = out[year].sum(skipna=True)
    out[f'%_{year}'] = np.where(total == 0, np.nan, out[year] / total * 100.0)
    return out, total

def horizontal_analysis(df, years_sorted):
    out = df[['Cuenta'] + years_sorted].copy()
    for i in range(1, len(years_sorted)):
        a, b = years_sorted[i-1], years_sorted[i]
        out[f'Var% {a}->{b}'] = np.where(out[a].fillna(0)==0, np.nan, (out[b]-out[a])/out[a]*100.0)
    return out

def pick_value(df, patterns, year, prefer_max=True):
    mask = pd.Series(False, index=df.index)
    for pat in patterns:
        looks_regex = bool(re.search(r'[.\^$\*\+\?{\[\]|()]', pat))
        if looks_regex:
            pat_nc = re.sub(r'\((?!\?)', '(?:', pat)
            m = df['Cuenta'].str.contains(pat_nc, case=False, regex=True, na=False)
        else:
            m = df['Cuenta'].str.contains(pat, case=False, regex=False, na=False)
        mask = mask | m
    vals = df.loc[mask, year]
    if vals.empty: return np.nan
    return vals.max(skipna=True) if prefer_max else vals.sum(skipna=True)

def sdiv(n, d):
    return np.nan if (d in [0, None] or pd.isna(d) or pd.isna(n)) else n/d

# ------------------------------------------------------------------------------
def load_core_reports(path):
    balance_wide, bal_years = parse_multi_year_sheet(path, 'ESTRUCTURA FINANCIERA')
    eres_wide,    er_years  = parse_multi_year_sheet(path, 'ESTRUCTURA ECONOMICA')
    balance_vertical_all = []
    for y in bal_years:
        vb, _ = vertical_analysis_exact(balance_wide, y, [r'^TOTAL\s+ACTIVOS?$'], prefer_max=True)
        balance_vertical_all.append(vb)
    er_vertical_all = []
    for y in er_years:
        ve, _ = vertical_analysis_exact(eres_wide, y, [r'^\s*\.?\s*=\s*INGRESOS\s+TOTALES\s*$', r'^\s*VENTAS\s*$', r'^\s*INGRESOS\s+NETOS\s*$'], prefer_max=True)
        er_vertical_all.append(ve)
    balance_horizontal = horizontal_analysis(balance_wide, bal_years)
    eres_horizontal    = horizontal_analysis(eres_wide,    er_years)
    years_common = sorted(set(bal_years).intersection(set(er_years)))
    ratios = pd.DataFrame(index=years_common)
    for y in years_common:
        AT  = pick_value(balance_wide, [r'^TOTAL\s+ACTIVOS?$'], y)
        PT  = pick_value(balance_wide, [r'^TOTAL\s+PASIVOS?$'], y)
        PAT = pick_value(balance_wide, [r'RECURSOS\s+PROPIOS', r'^PATRIMONIO$', r'CAPITAL\s+CONTABLE', r'FONDOS\s+PROPIOS'], y)
        AC  = pick_value(balance_wide, [r'^ACTIVO\s+(CIRCULANTE|CORRIENTE)$'], y)
        PC  = pick_value(balance_wide, [r'^PASIVO\s+(CIRCULANTE|CORRIENTE)$'], y)
        INV = pick_value(balance_wide, [r'INVENTARIOS?$', r'^EXISTENCIAS$'], y, prefer_max=False)
        VN  = pick_value(eres_wide, [r'^\s*\.?\s*=\s*INGRESOS\s+TOTALES\s*$', r'^\s*VENTAS\s*$', r'^\s*INGRESOS\s+NETOS\s*$'], y)
        UN  = pick_value(eres_wide, [r'UTILIDAD\s+NETA$', r'BENEFICIO\s+NETO$', r'GANANCIA\s+NETA$'], y)
        ratios.loc[y, 'Liquidez Corriente']                  = sdiv(AC, PC)
        ratios.loc[y, 'Prueba Ácida']                        = sdiv((AC - (0 if pd.isna(INV) else INV)), PC)
        ratios.loc[y, 'Endeudamiento (Pasivo/Activo)']       = sdiv(PT, AT)
        ratios.loc[y, 'Apalancamiento (Activo/Patrimonio)']  = sdiv(AT, PAT)
        ratios.loc[y, 'Margen Neto']                         = sdiv(UN, VN)
        ratios.loc[y, 'ROA']                                 = sdiv(UN, AT)
        ratios.loc[y, 'ROE']                                 = sdiv(UN, PAT)
        ratios.loc[y, 'Rotación de Activos']                 = sdiv(VN, AT)
    return balance_wide, bal_years, eres_wide, er_years, balance_vertical_all, er_vertical_all, balance_horizontal, eres_horizontal, ratios

def read_all_sheets(path):
    xls = pd.ExcelFile(path)
    sheets = {}
    for s in xls.sheet_names:
        try:
            sheets[s] = pd.read_excel(path, sheet_name=s)
        except Exception as e:
            sheets[s] = pd.DataFrame({'Error': [str(e)]})
    return sheets

def to_long_dataset(balance_wide, bal_years, eres_wide, er_years):
    b_long = balance_wide.melt(id_vars=['Cuenta'], value_vars=bal_years, var_name='Año', value_name='Monto')
    b_long['Reporte'] = 'Balance'
    e_long = eres_wide.melt(id_vars=['Cuenta'], value_vars=er_years, var_name='Año', value_name='Monto')
    e_long['Reporte'] = 'Resultados'
    out = pd.concat([b_long, e_long], ignore_index=True)
    return out[['Reporte','Cuenta','Año','Monto']]

# ------------------------------------------------------------------------------
FACTOR_CANDIDATES = ['Inflación_%','PIB_real_%','USD/DOP','TPM_%']
def fetch_wb_series(country_code, indicator_code, timeout_sec=6):
    url = f'https://api.worldbank.org/v2/country/{country_code}/indicator/{indicator_code}?format=json&per_page=20000'
    try:
        r = requests.get(url, timeout=timeout_sec)
        r.raise_for_status()
        data = r.json()
    except Exception:
        return pd.Series(dtype=float)
    if not isinstance(data, list) or len(data) < 2 or data[1] is None:
        return pd.Series(dtype=float)
    rows, out = data[1], {}
    for row in rows:
        yr, val = row.get('date'), row.get('value')
        if yr and val is not None:
            out[int(yr)] = float(val)
    return pd.Series(out).sort_index()

def build_external(years_all, country_code='DOM', tpm_dict=None):
    y0, y1 = min(years_all), max(years_all)
    ext = pd.DataFrame({'Año': range(y0-1, y1+1)})
    cpi_yoy   = fetch_wb_series(country_code, 'FP.CPI.TOTL.ZG')
    gdp_yoy   = fetch_wb_series(country_code, 'NY.GDP.MKTP.KD.ZG')
    fx_lcuusd = fetch_wb_series(country_code, 'PA.NUS.FCRF')
    ext['Inflación_%'] = ext['Año'].map(cpi_yoy.to_dict())
    ext['PIB_real_%']  = ext['Año'].map(gdp_yoy.to_dict())
    ext['USD/DOP']     = ext['Año'].map(fx_lcuusd.to_dict())
    if tpm_dict is None: tpm_dict = {}
    ext['TPM_%']       = ext['Año'].map(tpm_dict)
    return ext.set_index('Año').sort_index()

def _get_year_col(df, year):
    ys = str(year)
    for c in df.columns:
        if str(c).strip() == ys: return c
    return None

def pick_value_flex(df, patterns, year, prefer_max=True):
    if df is None or 'Cuenta' not in df.columns: return np.nan
    col = _get_year_col(df, year)
    if col is None: return np.nan
    mask = False
    for pat in patterns:
        mask = mask | df['Cuenta'].astype(str).str.contains(pat, case=False, regex=True, na=False)
    vals = df.loc[mask, col]
    if vals.empty: return np.nan
    return vals.max(skipna=True) if prefer_max else vals.sum(skipna=True)

def build_panel(years_all, eres_wide, ratios, external):
    ventas = pd.Series({y: pick_value_flex(eres_wide,[r'^\s*\.?\s*=\s*INGRESOS\s+TOTALES\s*$', r'^\s*VENTAS\s*$', r'^\s*INGRESOS\s+NETOS\s*$'],y) for y in years_all}, name='Ventas')
    util_neta = pd.Series({y: pick_value_flex(eres_wide,[r'UTILIDAD\s+NETA$', r'BENEFICIO\s+NETO$', r'GANANCIA\s+NETA$'],y) for y in years_all}, name='Utilidad Neta')
    panel = pd.DataFrame({'Ventas': ventas, 'Utilidad Neta': util_neta})
    rcopy = ratios.copy()
    try: rcopy.index = rcopy.index.astype(int)
    except: pass
    panel = panel.join(rcopy, how='left')
    ext_idx = external if external.index.name=='Año' else external.set_index('Año')
    panel = panel.join(ext_idx, how='left').sort_index()
    panel['Ventas_YoY_%']   = panel['Ventas'].astype(float).pct_change(periods=1, fill_method=None) * 100
    panel['UtilNeta_YoY_%'] = panel['Utilidad Neta'].astype(float).pct_change(periods=1, fill_method=None) * 100
    for col in ['Inflación_%','PIB_real_%','USD/DOP','TPM_%']:
        if col in panel.columns:
            panel[f'{col}_lag1'] = panel[col].shift(1)
    return panel

def zscore(s):
    s = s.astype(float); std = s.std(ddof=0)
    return (s - s.mean())/std if std and not np.isnan(std) and std != 0 else s*0

def ts_compare(panel_df, factor_col, kpi_col, lag=0, normalize=False):
    if factor_col not in panel_df.columns or kpi_col not in panel_df.columns:
        return go.Figure(), go.Figure(), pd.DataFrame(), 'Columnas inválidas.'
    fx = panel_df[factor_col].shift(lag) if lag else panel_df[factor_col]
    comp = pd.DataFrame({kpi_col: panel_df[kpi_col], factor_col: fx}).dropna()
    if comp.empty: return go.Figure(), go.Figure(), pd.DataFrame(), 'Sin datos tras filtros/lag.'
    kpi_plot = zscore(comp[kpi_col]) if normalize else comp[kpi_col]
    fac_plot = zscore(comp[factor_col]) if normalize else comp[factor_col]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=comp.index.astype(str), y=kpi_plot, mode='lines+markers', name=kpi_col))
    fig.add_trace(go.Scatter(x=comp.index.astype(str), y=fac_plot, mode='lines+markers', name=f"{factor_col}" + (f" (lag {lag})" if lag else ''), yaxis='y2'))
    fig.update_layout(title=f"{kpi_col} vs {factor_col}" + (f" (lag {lag})" if lag else ''), xaxis_title='Año', yaxis_title=kpi_col + (' (z-score)' if normalize else ''), yaxis2=dict(title=factor_col + (' (z-score)' if normalize else ''), overlaying='y', side='right'), template='plotly_white')
    scatter = go.Figure(); x, y = comp[factor_col].values, comp[kpi_col].values
    scatter.add_trace(go.Scatter(x=x, y=y, mode='markers+text', text=comp.index.astype(str), textposition='top center', name='Obs'))
    try:
        coef = np.polyfit(x, y, 1); x_fit = np.linspace(x.min(), x.max(), 50)
        y_fit = coef[0]*x_fit + coef[1]; y_hat = coef[0]*x + coef[1]
        r2 = 1 - np.sum((y - y_hat)**2)/np.sum((y - y.mean())**2)
        scatter.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', name=f'Ajuste (R²={r2:.3f})'))
    except Exception:
        pass
    scatter.update_layout(title=f'Dispersión: {kpi_col} vs {factor_col}', xaxis_title=factor_col, yaxis_title=kpi_col, template='plotly_white')
    corr = comp[kpi_col].corr(comp[factor_col])
    info = f"**Correlación (Pearson)** {kpi_col} vs {factor_col}" + (f" (lag {lag})" if lag else '') + f": **{corr:.3f}**"
    return fig, scatter, comp.reset_index(names='Año'), info

def corr_heatmap(panel_df, lag=0):
    KPI_COLS = [c for c in ['Ventas_YoY_%','UtilNeta_YoY_%','Margen Neto','ROE','ROA','Rotación de Activos'] if c in panel_df.columns]
    FACTORS  = [c for c in FACTOR_CANDIDATES if c in panel_df.columns]
    if not KPI_COLS or not FACTORS: return go.Figure(), pd.DataFrame()
    mat = pd.DataFrame(index=KPI_COLS, columns=FACTORS, dtype=float)
    for k in KPI_COLS:
        for f in FACTORS:
            s = pd.concat([panel_df[k], panel_df[f].shift(lag) if lag else panel_df[f]], axis=1).dropna()
            mat.loc[k, f] = s.iloc[:,0].corr(s.iloc[:,1])
    fig = go.Figure(data=go.Heatmap(z=mat.values, x=mat.columns.tolist(), y=mat.index.tolist(), zmin=-1, zmax=1, colorscale='RdBu'))
    fig.update_layout(title=f'Heatmap de correlaciones (lag factores = {lag})', template='plotly_white')
    return fig, mat.reset_index()

def run_ols(panel_df, dep, indeps):
    if dep not in panel_df.columns or not indeps:
        return 'Selecciona dependiente y factores.'
    df = panel_df[[dep] + indeps].dropna().copy()
    if df.empty or df.shape[0] < len(indeps)+2:
        return 'No hay suficientes observaciones.'
    y = df[dep].astype(float); X = sm.add_constant(df[indeps].astype(float), has_constant='add')
    model = sm.OLS(y, X).fit()
    out = pd.DataFrame({'Variable': ['Constante'] + indeps,'Coef': model.params.values.round(4),'StdErr': model.bse.values.round(4),'t': model.tvalues.values.round(3),'p>|t|': model.pvalues.values.round(4)})
    meta = f'R²={model.rsquared:.3f} | R² ajustado={model.rsquared_adj:.3f} | N={int(model.nobs)}'
    out_csv = os.path.join(OUTPUT_DIR, 'ols_results.csv'); out.to_csv(out_csv, index=False)
    return out, meta, out_csv

def generate_pdf(balance_wide, bal_years, eres_wide, er_years, ratios):
    yrs = sorted(set(map(int, bal_years)).intersection(set(map(int, er_years))))
    if not yrs: return None
    def _get_series(df, pats):
        return pd.Series({y: pick_value_flex(df, pats, y) for y in yrs})
    VN  = _get_series(eres_wide, [r'^\s*\.?\s*=\s*INGRESOS\s+TOTALES\s*$', r'^\s*VENTAS\s*$', r'^\s*INGRESOS\s+NETOS\s*$'])
    UN  = _get_series(eres_wide, [r'UTILIDAD\s+NETA$', r'BENEFICIO\s+NETO$', r'GANANCIA\s+NETA$'])
    AT  = _get_series(balance_wide, [r'^TOTAL\s+ACTIVOS?$'])
    PT  = _get_series(balance_wide, [r'^TOTAL\s+PASIVOS?$'])
    PAT = _get_series(balance_wide, [r'RECURSOS\s+PROPIOS', r'^PATRIMONIO$', r'CAPITAL\s+CONTABLE', r'FONDOS\s+PROPIOS'])
    def save_line_png(series, title, fname, ylabel=''):
        s = pd.Series(series).dropna()
        if s.empty: return None
        plt.figure(); plt.plot(list(map(str, s.index)), s.values, marker='o')
        plt.title(title); plt.xlabel('Año'); plt.ylabel(ylabel); plt.grid(True)
        out = os.path.join(OUTPUT_DIR, f'{fname}.png'); plt.savefig(out, bbox_inches='tight', dpi=140); plt.close(); return out
    imgs = []
    for s, title, fn, yl in [(VN,'Ingresos/Ventas','plot_ventas','Monto'),(UN,'Utilidad Neta','plot_utilidad','Monto'),(AT,'Total Activos','plot_activos','Monto'),(PT,'Total Pasivos','plot_pasivos','Monto'),(PAT,'Patrimonio','plot_patrimonio','Monto')]:
        p = save_line_png(s, title, fn, yl);  imgs.append(p) if p else None
    if 'Liquidez Corriente' in ratios.columns:
        p = save_line_png(ratios['Liquidez Corriente'], 'Liquidez Corriente', 'plot_liquidez', 'Veces'); imgs.append(p) if p else None
    if 'Endeudamiento (Pasivo/Activo)' in ratios.columns:
        p = save_line_png(ratios['Endeudamiento (Pasivo/Activo)'], 'Endeudamiento', 'plot_endeuda', 'Proporción'); imgs.append(p) if p else None
    if 'Margen Neto' in ratios.columns:
        p = save_line_png(ratios['Margen Neto'], 'Margen Neto', 'Proporción'); imgs.append(p) if p else None
    pdf_path = os.path.join(OUTPUT_DIR, 'analisis_financiero.pdf')
    doc = SimpleDocTemplate(pdf_path, pagesize=landscape(letter), leftMargin=24, rightMargin=24, topMargin=24, bottomMargin=24)
    styles = getSampleStyleSheet(); H1, H2, BODY = styles['Title'], styles['Heading2'], styles['BodyText']
    story = []
    story.append(Paragraph('📊 Análisis Financiero — Resumen', H1)); story.append(Spacer(1, 8))
    story.append(Paragraph(f'Años Balance: {", ".join(bal_years)}', BODY))
    story.append(Paragraph(f'Años E. Resultados: {", ".join(er_years)}', BODY))
    try:
        g_ing = (VN.iloc[-1]/VN.iloc[0]-1)*100 if VN.iloc[0] else np.nan
        g_un  = (UN.iloc[-1]/UN.iloc[0]-1)*100 if UN.iloc[0] else np.nan
        liq   = ratios.iloc[-1]['Liquidez Corriente'] if 'Liquidez Corriente' in ratios.columns else np.nan
        ende  = ratios.iloc[-1]['Endeudamiento (Pasivo/Activo)'] if 'Endeudamiento (Pasivo/Activo)' in ratios.columns else np.nan
        story.append(Spacer(1, 6))
        story.append(Paragraph(f'<b>Resumen:</b> Ingresos {g_ing:,.2f}% desde {yrs[0]} a {yrs[-1]}; Utilidad Neta {g_un:,.2f}%. Liquidez Corriente {liq:,.2f}×; Endeudamiento {ende*100:,.2f}%.', BODY))
    except Exception:
        pass
    if imgs:
        story.append(PageBreak()); story.append(Paragraph('Gráficos de tendencia', H2))
        for p in imgs: story.append(Image(p, width=520, height=280)); story.append(Spacer(1,6))
    def df_to_tabledata(df, max_rows=28):
        d = df.copy()
        if d.index.name or not isinstance(d.index, pd.RangeIndex): d = d.reset_index()
        if len(d) > max_rows: d = d.head(max_rows)
        header = [str(c) for c in d.columns.tolist()]
        def is_pct_col(colname):
            return (str(colname).startswith('%_') or str(colname).startswith('Var%') or str(colname) in ['Margen Neto','ROA','ROE','Endeudamiento (Pasivo/Activo)'])
        rows = []
        for _, row in d.iterrows():
            r = []
            for c in d.columns:
                v = row[c]
                if pd.isna(v): r.append('')
                elif isinstance(v,(int,float)):
                    r.append(f'{v*100:,.2f}%' if (is_pct_col(c) and abs(v)<=2) else f'{v:,.2f}')
                else:
                    r.append(str(v))
            rows.append(r)
        return [header] + rows
    def styled_table(data, colWidths=None):
        t = Table(data, colWidths=colWidths)
        t.setStyle(TableStyle([('BACKGROUND', (0,0), (-1,0), colors.HexColor('#4a5568')),('TEXTCOLOR',  (0,0), (-1,0), colors.whitesmoke),('ALIGN',(0,0),(-1,0),'CENTER'),('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),('FONTSIZE',(0,0),(-1,0),9),('BOTTOMPADDING',(0,0),(-1,0),6),('ALIGN',(0,1),(-1,-1),'RIGHT'),('ALIGN',(0,1),(0,-1),'LEFT'),('FONTSIZE',(0,1),(-1,-1),8),('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.whitesmoke, colors.HexColor('#f7fafc')]),('GRID',(0,0),(-1,-1),0.25,colors.grey)]))
        return t
    bal_v_all = []; er_v_all = []
    for y in bal_years:
        v,_ = vertical_analysis_exact(balance_wide, y, [r'^TOTAL\s+ACTIVOS?$']); bal_v_all.append(v)
    for y in er_years:
        v,_ = vertical_analysis_exact(eres_wide, y, [r'^\s*\.?\s*=\s*INGRESOS\s+TOTALES\s*$', r'^\s*VENTAS\s*$', r'^\s*INGRESOS\s+NETOS\s*$']); er_v_all.append(v)
    for i, y in enumerate(bal_years):
        story.append(PageBreak()); story.append(Paragraph(f'Análisis Vertical — Balance General ({y})', H2))
        story.append(styled_table(df_to_tabledata(bal_v_all[i], max_rows=28), colWidths=[280, 140, 120]))
    for i, y in enumerate(er_years):
        story.append(PageBreak()); story.append(Paragraph(f'Análisis Vertical — Estado de Resultados ({y})', H2))
        story.append(styled_table(df_to_tabledata(er_v_all[i], max_rows=28), colWidths=[280, 140, 120]))
    story.append(PageBreak()); story.append(Paragraph('Análisis Horizontal — Balance General', H2))
    story.append(styled_table(df_to_tabledata(horizontal_analysis(balance_wide, bal_years), max_rows=32)))
    story.append(PageBreak()); story.append(Paragraph('Análisis Horizontal — Estado de Resultados', H2))
    story.append(styled_table(df_to_tabledata(horizontal_analysis(eres_wide, er_years), max_rows=32)))
    story.append(PageBreak()); story.append(Paragraph('Ratios Financieros por Año', H2))
    rat_out = ratios.copy(); rat_out.index.name = 'Año'
    story.append(styled_table(df_to_tabledata(rat_out, max_rows=9999)))
    doc.build(story)
    return pdf_path

# ------------------------------------------------------------------------------
def build_app(default_path=DEFAULT_PATH):
    with gr.Blocks(title='Dashboard Financiero — Unificado v6') as demo:
        gr.Markdown('## 📊 Dashboard Financiero — Unificado (Excel ⇢ Dashboard ⇢ Power BI ⇢ PDF)')
        with gr.Row():
            path_tb = gr.Textbox(value=default_path, label='Ruta del Excel (Drive)')
            file_up = gr.File(label='O sube el Excel (.xlsx/.xls)', file_types=['.xlsx','.xls'])
            load_btn = gr.Button('Cargar / Refrescar')
        info_md = gr.Markdown()
        with gr.Tab('Explorar Excel'):
            sheet_dd = gr.Dropdown(choices=[], label='Hoja')
            sheet_df = gr.Dataframe(interactive=False, label='Vista de la hoja')
            dl_sheet = gr.File(label='Descargar CSV de la hoja seleccionada')
        with gr.Tab('Balance / E.R. (Vertical & Horizontal)'):
            year_bal_dd = gr.Dropdown(choices=[], label='Año Balance (vertical)')
            vbal_plot = gr.Plot(); vbal_tbl = gr.Dataframe(interactive=False)
            year_er_dd = gr.Dropdown(choices=[], label='Año E.R. (vertical)')
            ver_plot = gr.Plot(); ver_tbl = gr.Dataframe(interactive=False)
            bal_h_tbl = gr.Dataframe(interactive=False, label='Balance Horizontal (YoY)')
            er_h_tbl  = gr.Dataframe(interactive=False, label='E.R. Horizontal (YoY)')
        with gr.Tab('Ratios'):
            ratio_dd   = gr.Dropdown(choices=[], label='Selecciona un ratio')
            ratio_plot = gr.Plot(); ratio_tbl  = gr.Dataframe(interactive=False)
        with gr.Tab('Dataset Power BI'):
            pbi_df   = gr.Dataframe(interactive=False)
            pbi_file = gr.File()
        with gr.Tab('1) Factores externos'):
            with gr.Row():
                cc_in = gr.Textbox(value='DOM', label='Código de país (ISO3)')
                btn_wdi = gr.Button('Actualizar WDI')
            wdi_info = gr.Markdown(); ext_tbl  = gr.Dataframe(interactive=False, label='Factores externos'); pan_tbl  = gr.Dataframe(interactive=False, label='Panel interno–externo')
            tpm_editor = gr.Dataframe(headers=['Año','TPM_%'], interactive=True, label='Editor TPM (anual)'); btn_save_tpm = gr.Button('Guardar TPM')
            up_file = gr.File(label='Subir CSV externo', file_types=['.csv']); btn_load_csv = gr.Button('Cargar CSV externo')
            dl_ext = gr.File(); dl_pan = gr.File()
        with gr.Tab('2) Series comparadas'):
            factor_dd = gr.Dropdown(choices=[], label='Factor externo'); kpi_dd = gr.Dropdown(choices=[], label='KPI interno')
            lag_sl = gr.Slider(0, 2, value=0, step=1, label='Rezago (años)'); norm_cb = gr.Checkbox(False, label='Normalizar (z-score)')
            ts_plot = gr.Plot(); sc_plot = gr.Plot(); ts_table = gr.Dataframe(interactive=False); info_cmp = gr.Markdown()
        with gr.Tab('3) Heatmap correlaciones'):
            lag_heat = gr.Slider(0, 2, value=0, step=1, label='Rezago factores'); heat_plot = gr.Plot(); heat_tbl  = gr.Dataframe(interactive=False)
        with gr.Tab('4) Regresión OLS'):
            dep_dd = gr.Dropdown(choices=[], label='Dependiente (KPI)')
            indep_cg = gr.CheckboxGroup(choices=[], label='Factores (elige varios)')
            ols_tbl = gr.Dataframe(interactive=False); ols_meta = gr.Markdown(); ols_file = gr.File()
        with gr.Tab('5) PESTEL'):
            year_dd = gr.Dropdown(choices=[], label='Año de evaluación')
            with gr.Row():
                econ_w = gr.Slider(0, 100, value=25, step=1, label='Económico'); soc_w = gr.Slider(0,100,value=10,step=1,label='Social'); geo_w = gr.Slider(0,100,value=5,step=1,label='Geográfico')
            with gr.Row():
                pol_w = gr.Slider(0, 100, value=25, step=1, label='Político'); tec_w = gr.Slider(0,100,value=25,step=1,label='Tecnológico'); cul_w = gr.Slider(0,100,value=10,step=1,label='Cultural')
            auto_econ = gr.Checkbox(True, label='Auto Económico (PIB, Inflación, Depreciación, TPM)'); econ_manual= gr.Slider(-100, 100, value=0, step=1, label='Económico (manual)', interactive=True)
            econ_auto_out = gr.Number(label='Económico (auto)', interactive=False)
            soc_res = gr.Slider(-100, 100, value=0, step=1, label='Social'); geo_res = gr.Slider(-100,100,value=0,step=1,label='Geográfico')
            pol_res = gr.Slider(-100, 100, value=0, step=1, label='Político'); tec_res = gr.Slider(-100,100,value=0,step=1,label='Tecnológico'); cul_res = gr.Slider(-100,100,value=0,step=1,label='Cultural')
            idx_plot = gr.Plot(); idx_tbl  = gr.Dataframe(interactive=False); idx_md   = gr.Markdown(); idx_file = gr.File()
        with gr.Tab('6) Exportar PDF'):
            pdf_file = gr.File(label='PDF generado'); gen_pdf_btn = gr.Button('Generar PDF')
        state = gr.State({})

        def do_load(path_text, file_obj):
            t0 = time.time()
            try:
                path = None
                if file_obj is not None:
                    path = file_obj if isinstance(file_obj, str) else getattr(file_obj, 'name', None)
                elif path_text and os.path.exists(path_text):
                    path = path_text
                else:
                    raise FileNotFoundError('Especifica una ruta válida o sube el archivo .xlsx')
                sheets = read_all_sheets(path)
                balance_wide, bal_years, eres_wide, er_years, bal_v_all, er_v_all, bal_h, er_h, ratios = load_core_reports(path)
                pbi_long = to_long_dataset(balance_wide, bal_years, eres_wide, er_years)
                pbi_path = os.path.join(OUTPUT_DIR, 'powerbi_dataset_long.csv'); pbi_long.to_csv(pbi_path, index=False)
                years_all = sorted({*map(int, bal_years), *map(int, er_years)})
                # Carga rápida: no llamar APIs externas aquí; se actualiza con el botón "Actualizar WDI".
                external_df = pd.DataFrame({'Año': range(min(years_all)-1, max(years_all)+1)}).set_index('Año')
                for c in FACTOR_CANDIDATES:
                    external_df[c] = np.nan
                panel_df    = build_panel(years_all, eres_wide, ratios, external_df)
                # Evitar exportaciones pesadas en la carga inicial.
                sheet_exports = {}
                st = {'path': path,'sheets': sheets,'sheet_exports': sheet_exports,'balance_wide': balance_wide,'bal_years': bal_years,'eres_wide': eres_wide,'er_years': er_years,'bal_v_all': bal_v_all,'er_v_all': er_v_all,'bal_h': bal_h,'er_h': er_h,'ratios': ratios,'pbi_long': pbi_long,'pbi_path': pbi_path,'external_df': external_df,'panel_df': panel_df,'years_all': years_all}
                elapsed = time.time() - t0
                info = f"✅ Archivo cargado: {os.path.basename(path)} — Hojas: {len(sheets)} — {elapsed:.2f}s"
                return (st, info, gr.update(choices=list(sheets.keys()), value=list(sheets.keys())[0] if sheets else None), sheets[list(sheets.keys())[0]] if sheets else pd.DataFrame(), None, gr.update(choices=bal_years, value=(bal_years[0] if bal_years else None)), go.Figure(), pd.DataFrame(), gr.update(choices=er_years, value=(er_years[0] if er_years else None)), go.Figure(), pd.DataFrame(), bal_h, er_h, gr.update(choices=list(ratios.columns), value=(list(ratios.columns)[0] if len(ratios.columns)>0 else None)), go.Figure(), ratios.reset_index().rename(columns={'index':'Año'}) if not ratios.empty else pd.DataFrame(), pbi_long, pbi_path, external_df.reset_index(), panel_df.reset_index(), gr.update(value=pd.DataFrame({'Año': external_df.index, 'TPM_%': external_df.get('TPM_%')})), external_df.reset_index(), panel_df.reset_index(), os.path.join(OUTPUT_DIR, 'factores_externos_descarga.csv'), os.path.join(OUTPUT_DIR, 'panel_interno_externo_descarga.csv'), gr.update(choices=[c for c in FACTOR_CANDIDATES if c in panel_df.columns], value=None), gr.update(choices=[c for c in ['Ventas_YoY_%','UtilNeta_YoY_%','Margen Neto','ROE','ROA','Rotación de Activos'] if c in panel_df.columns], value=None), gr.update(choices=panel_df.index.tolist(), value=(panel_df.index.tolist()[-1] if len(panel_df.index)>0 else None)), gr.update(choices=[c for c in ['Ventas_YoY_%','UtilNeta_YoY_%','Margen Neto','ROE','ROA','Rotación de Activos'] if c in panel_df.columns], value=None), gr.update(choices=[c for c in FACTOR_CANDIDATES if c in panel_df.columns] + [f"{c}_lag1" for c in FACTOR_CANDIDATES if f"{c}_lag1" in panel_df.columns], value=[]))
            except Exception as e:
                tb = traceback.format_exc()
                return ({}, f'❌ Error: {e}\n\n{tb}', gr.update(choices=[]), pd.DataFrame(), None, gr.update(choices=[]), go.Figure(), pd.DataFrame(), gr.update(choices=[]), go.Figure(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), gr.update(choices=[]), go.Figure(), pd.DataFrame(), pd.DataFrame(), None, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), None, None, gr.update(choices=[]), gr.update(choices=[]), gr.update(choices=[]), gr.update(choices=[]), gr.update(choices=[]))
        load_btn.click(do_load, inputs=[path_tb, file_up], outputs=[state, info_md, sheet_dd, sheet_df, dl_sheet, year_bal_dd, vbal_plot, vbal_tbl, year_er_dd, ver_plot, ver_tbl, bal_h_tbl, er_h_tbl, ratio_dd, ratio_plot, ratio_tbl, pbi_df, pbi_file, ext_tbl, pan_tbl, tpm_editor, ext_tbl, pan_tbl, dl_ext, dl_pan, factor_dd, kpi_dd, year_dd, dep_dd, indep_cg])

        def show_sheet(sname, state_dict):
            if not state_dict: return pd.DataFrame(), None
            sheets = state_dict['sheets']
            exps = state_dict.get('sheet_exports', {})
            if sname in sheets:
                df = sheets[sname]
                out_csv = exps.get(sname)
                if out_csv is None:
                    sheet_export_dir = os.path.join(OUTPUT_DIR, 'sheet_exports')
                    os.makedirs(sheet_export_dir, exist_ok=True)
                    out_csv = os.path.join(sheet_export_dir, f"{re.sub(r'[^A-Za-z0-9_-]+','_',sname)}.csv")
                    try:
                        df.to_csv(out_csv, index=False)
                    except Exception:
                        df.to_csv(out_csv)
                    exps[sname] = out_csv
                    state_dict['sheet_exports'] = exps
                return df, out_csv
            return pd.DataFrame(), None
        sheet_dd.change(show_sheet, inputs=[sheet_dd, state], outputs=[sheet_df, dl_sheet])

        def _view_vbal(y, state_dict):
            if not state_dict: return go.Figure(), pd.DataFrame()
            bal_years = state_dict['bal_years']; bal_v_all = state_dict['bal_v_all']
            if y not in bal_years: return go.Figure(), pd.DataFrame()
            idx = bal_years.index(y); df = bal_v_all[idx]
            col_pct = f'%_{y}'
            tmp = df[['Cuenta', col_pct]].dropna(); tmp = tmp[~tmp['Cuenta'].astype(str).str.contains('TOTAL', case=False, na=False)]
            tmp = tmp.sort_values(col_pct, ascending=False).head(10)
            fig = go.Figure(go.Bar(x=tmp[col_pct].values, y=tmp['Cuenta'].values, orientation='h'))
            fig.update_layout(title=f'Top 10 partidas por porcentaje ({y})', xaxis_title='Porcentaje (%)', yaxis_title='', template='plotly_white')
            return fig, df
        year_bal_dd.change(_view_vbal, inputs=[year_bal_dd, state], outputs=[vbal_plot, vbal_tbl])

        def _view_ver(y, state_dict):
            if not state_dict: return go.Figure(), pd.DataFrame()
            er_years = state_dict['er_years']; er_v_all = state_dict['er_v_all']
            if y not in er_years: return go.Figure(), pd.DataFrame()
            idx = er_years.index(y); df = er_v_all[idx]
            col_pct = f'%_{y}'
            tmp = df[['Cuenta', col_pct]].dropna(); tmp = tmp[~tmp['Cuenta'].astype(str).str.contains('TOTAL', case=False, na=False)]
            tmp = tmp.sort_values(col_pct, ascending=False).head(10)
            fig = go.Figure(go.Bar(x=tmp[col_pct].values, y=tmp['Cuenta'].values, orientation='h'))
            fig.update_layout(title=f'Top 10 conceptos por porcentaje ({y})', xaxis_title='Porcentaje (%)', yaxis_title='', template='plotly_white')
            return fig, df
        year_er_dd.change(_view_ver, inputs=[year_er_dd, state], outputs=[ver_plot, ver_tbl])

        def _ratio_plot(rname, state_dict):
            if not state_dict: return go.Figure()
            ratios = state_dict['ratios']
            if rname in ratios.columns:
                s = ratios[rname]; fig = go.Figure(); fig.add_trace(go.Scatter(x=list(map(str, s.index)), y=s.values, mode='lines+markers', name=rname))
                fig.update_layout(title=rname, xaxis_title='Año', yaxis_title='Valor', template='plotly_white')
                return fig
            return go.Figure()
        ratio_dd.change(_ratio_plot, inputs=[ratio_dd, state], outputs=[ratio_plot])

        def refresh_wdi(cc, state_dict):
            if not state_dict: return 'Carga primero el Excel.', pd.DataFrame(), pd.DataFrame(), gr.update(), gr.update()
            years_all = state_dict['years_all']
            try:
                tpm_dict = state_dict['external_df']['TPM_%'].dropna().to_dict() if 'TPM_%' in state_dict['external_df'].columns else {}
                external_df = build_external(years_all, country_code=cc, tpm_dict=tpm_dict)
                panel_df    = build_panel(years_all, state_dict['eres_wide'], state_dict['ratios'], external_df)
                state_dict['external_df'] = external_df; state_dict['panel_df'] = panel_df
                ext_csv = os.path.join(OUTPUT_DIR, 'factores_externos_descarga.csv'); external_df.to_csv(ext_csv)
                pan_csv = os.path.join(OUTPUT_DIR, 'panel_interno_externo_descarga.csv'); panel_df.to_csv(pan_csv)
                msg = f'Actualizado WDI para {cc}.'
                if external_df[['Inflación_%','PIB_real_%','USD/DOP']].dropna(how='all').empty:
                    msg = f'Sin conexión o sin datos WDI para {cc}. Se mantiene panel local.'
                return (msg, external_df.reset_index(), panel_df.reset_index(), ext_csv, pan_csv)
            except Exception as e:
                return (f'Error WDI: {e}', state_dict['external_df'].reset_index(), state_dict['panel_df'].reset_index(), gr.update(), gr.update())
        btn_wdi.click(refresh_wdi, inputs=[cc_in, state], outputs=[wdi_info, ext_tbl, pan_tbl, dl_ext, dl_pan])

        def save_tpm(tpm_df_, state_dict):
            if not state_dict: return 'Carga primero el Excel.', pd.DataFrame(), pd.DataFrame()
            try:
                tmp = pd.DataFrame(tpm_df_, columns=['Año','TPM_%']).dropna(subset=['Año']); tmp['Año'] = tmp['Año'].astype(int)
                ext_new = state_dict['external_df'].copy()
                for _, r in tmp.iterrows():
                    ext_new.loc[int(r['Año']), 'TPM_%'] = float(r['TPM_%']) if pd.notna(r['TPM_%']) else np.nan
                state_dict['external_df'] = ext_new.sort_index()
                panel_df = build_panel(state_dict['years_all'], state_dict['eres_wide'], state_dict['ratios'], state_dict['external_df'])
                state_dict['panel_df'] = panel_df
                return ('TPM guardada y panel actualizado.', state_dict['external_df'].reset_index(), panel_df.reset_index())
            except Exception as e:
                return (f'Error guardando TPM: {e}', state_dict['external_df'].reset_index(), state_dict['panel_df'].reset_index())
        btn_save_tpm.click(save_tpm, inputs=[tpm_editor, state], outputs=[wdi_info, ext_tbl, pan_tbl])

        def load_external_csv(fileobj, state_dict):
            if not state_dict: return 'Carga primero el Excel.', pd.DataFrame(), pd.DataFrame()
            if fileobj is None:
                return 'Sube un CSV primero.', state_dict['external_df'].reset_index(), state_dict['panel_df'].reset_index()
            try:
                csv_path = fileobj if isinstance(fileobj, str) else getattr(fileobj, 'name', None)
                if not csv_path:
                    return 'No se pudo leer el archivo CSV.', state_dict['external_df'].reset_index(), state_dict['panel_df'].reset_index()
                df = pd.read_csv(csv_path)
                if 'Año' not in df.columns:
                    return "El CSV debe contener la columna 'Año'.", state_dict['external_df'].reset_index(), state_dict['panel_df'].reset_index()
                df['Año'] = df['Año'].astype(int); df = df.set_index('Año').sort_index()
                cols_ok = [c for c in ['Inflación_%','PIB_real_%','USD/DOP','TPM_%'] if c in df.columns]
                if not cols_ok:
                    return 'No se encontraron columnas externas válidas en el CSV.', state_dict['external_df'].reset_index(), state_dict['panel_df'].reset_index()
                ext_new = state_dict['external_df'].copy(); ext_new.update(df[cols_ok]); state_dict['external_df'] = ext_new.sort_index()
                panel_df = build_panel(state_dict['years_all'], state_dict['eres_wide'], state_dict['ratios'], state_dict['external_df'])
                state_dict['panel_df'] = panel_df
                return f'CSV cargado. Columnas: {", ".join(cols_ok)}', state_dict['external_df'].reset_index(), panel_df.reset_index()
            except Exception as e:
                return f'Error leyendo CSV: {e}', state_dict['external_df'].reset_index(), state_dict['panel_df'].reset_index()
        btn_load_csv.click(load_external_csv, inputs=[up_file, state], outputs=[wdi_info, ext_tbl, pan_tbl])

        def _ts(factor, kpi, lag, norm, state_dict):
            if not state_dict: return go.Figure(), go.Figure(), pd.DataFrame(), 'Carga primero el Excel.'
            return ts_compare(state_dict['panel_df'], factor, kpi, lag=int(lag), normalize=bool(norm))
        gr.Button('Comparar').click(_ts, inputs=[factor_dd, kpi_dd, lag_sl, norm_cb, state], outputs=[ts_plot, sc_plot, ts_table, info_cmp])

        def _heat(lag, state_dict):
            if not state_dict: return go.Figure(), pd.DataFrame()
            return corr_heatmap(state_dict['panel_df'], lag=int(lag))
        gr.Button('Generar').click(_heat, inputs=[lag_heat, state], outputs=[heat_plot, heat_tbl])

        def _ols(dep, indeps, state_dict):
            if not state_dict: return pd.DataFrame(), 'Carga primero el Excel.', None
            res = run_ols(state_dict['panel_df'], dep, indeps)
            if isinstance(res, str): return pd.DataFrame(), res, None
            tbl, meta, path = res; return tbl, meta, path
        gr.Button('Ejecutar OLS').click(_ols, inputs=[dep_dd, indep_cg, state], outputs=[ols_tbl, ols_meta, ols_file])

        def _econ_auto(year, e_w, s_w, g_w, p_w, t_w, c_w, auto_e, e_man, s_r, g_r, p_r, t_r, c_r, state_dict):
            ext = state_dict.get('external_df', pd.DataFrame()); year = int(year)
            def _scale_to_score(x, lo, hi, pos_good=True):
                if x is None or (isinstance(x, float) and np.isnan(x)): return np.nan
                x_clip = min(max(x, lo), hi); u = (x_clip - lo) / (hi - lo + 1e-9)
                if not pos_good: u = 1 - u
                return (u * 2 - 1) * 100
            econ_auto = None
            if isinstance(ext, pd.DataFrame) and not ext.empty:
                try:
                    gdp = ext.at[year, 'PIB_real_%'] if 'PIB_real_%' in ext.columns and year in ext.index else np.nan
                    inf = ext.at[year, 'Inflación_%'] if 'Inflación_%' in ext.columns and year in ext.index else np.nan
                    tpm = ext.at[year, 'TPM_%']       if 'TPM_%' in ext.columns and year in ext.index else np.nan
                    fx_score = np.nan
                    if 'USD/DOP' in ext.columns and year in ext.index and (year-1) in ext.index:
                        fx, fx_prev = ext.at[year, 'USD/DOP'], ext.at[year-1, 'USD/DOP']
                        dep_yoy = (fx/fx_prev - 1) * 100 if fx_prev not in (0, np.nan, None) else np.nan
                        fx_score = _scale_to_score(dep_yoy, lo=-5, hi=30, pos_good=False)
                    s_gdp = _scale_to_score(gdp, lo=-5, hi=10, pos_good=True)
                    s_inf = _scale_to_score(inf, lo=0,  hi=20, pos_good=False)
                    s_tpm = _scale_to_score(tpm, lo=0,  hi=20, pos_good=False)
                    vals = [v for v in [s_gdp, s_inf, fx_score, s_tpm] if not np.isnan(v)]
                    econ_auto = float(np.mean(vals)) if vals else None
                except Exception:
                    econ_auto = None
            econ_val = econ_auto if auto_e else float(e_man)
            pesos = {'Económico':e_w, 'Social':s_w, 'Geográfico':g_w, 'Político':p_w, 'Tecnológico':t_w, 'Cultural':c_w}
            w_sum = sum(pesos.values()) or 1.0; pesos_norm = {k: float(v)/w_sum for k,v in pesos.items()}
            res_map = {'Económico':econ_val, 'Social':s_r, 'Geográfico':g_r, 'Político':p_r, 'Tecnológico':t_r, 'Cultural':c_r}
            rows=[]; total=0.0
            for k in ['Económico','Social','Geográfico','Político','Tecnológico','Cultural']:
                val = float(res_map[k]) if res_map[k] is not None else 0.0
                aporte = pesos_norm[k]*val; rows.append([k, pesos[k], pesos_norm[k]*100, val, aporte]); total += aporte
            df = pd.DataFrame(rows, columns=['Categoría','Peso % (input)','Peso % (normalizado)','Resultado %','Aporte ponderado %'])
            clas = 'Excelente' if total>15 else 'Bueno' if total>5 else 'Neutral' if total>-5 else 'Crítico' if total>-15 else 'Muy crítico'
            fig = go.Figure(); fig.add_trace(go.Bar(x=df['Categoría'], y=df['Resultado %'], name='Resultado (%)'))
            fig.add_trace(go.Bar(x=df['Categoría'], y=df['Aporte ponderado %'], name='Aporte ponderado (pp)'))
            fig.update_layout(barmode='group', title=f'Índice de Entorno ({year}) — Total {total:.1f}% · {clas}', template='plotly_white')
            out_csv = os.path.join(OUTPUT_DIR, 'indice_entorno.csv'); df.assign(Total_Indice=total, Clasificacion=clas).to_csv(out_csv, index=False)
            return fig, df, f'**Índice total {year}: {total:.1f}% → {clas}**', out_csv, econ_val
        gr.Button('Calcular índice').click(_econ_auto, inputs=[year_dd, econ_w, soc_w, geo_w, pol_w, tec_w, cul_w, auto_econ, econ_manual, soc_res, geo_res, pol_res, tec_res, cul_res, state], outputs=[idx_plot, idx_tbl, idx_md, idx_file, econ_auto_out])

        def _gen_pdf(state_dict):
            if not state_dict: return None
            path = generate_pdf(state_dict['balance_wide'], state_dict['bal_years'], state_dict['eres_wide'], state_dict['er_years'], state_dict['ratios'])
            return path
        gen_pdf_btn.click(_gen_pdf, inputs=[state], outputs=[pdf_file])

    return demo

app = build_app()
app.launch(share=False, debug=False)

Task exception was never retrieved
future: <Task finished coro=<Queue.process_events() done, defined at /Users/juanjo/Library/Python/3.7/lib/python/site-packages/gradio/queueing.py:343> exception=1 validation error for PredictBody
event_id
  Field required [type=missing, input_value={'data': ['/Users/juanjo/...ion_hash': 'qf32m8yiye'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.5/v/missing>
Traceback (most recent call last):
  File "/Users/juanjo/Library/Python/3.7/lib/python/site-packages/gradio/queueing.py", line 347, in process_events
    client_awake = await self.gather_event_data(event)
  File "/Users/juanjo/Library/Python/3.7/lib/python/site-packages/gradio/queueing.py", line 220, in gather_event_data
    data, client_awake = await self.get_message(event, timeout=receive_timeout)
  File "/Users/juanjo/Library/Python/3.7/lib/python/site-packages/gradio/queueing.py", line 456, in get_message
    return PredictBody(**data), True
  File "/Users/j

Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


In [8]:
import importlib

required = ['gradio', 'plotly', 'openpyxl', 'statsmodels', 'reportlab', 'requests']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print('Faltan paquetes. Instálalos con:')
    print('pip install ' + ' '.join(missing))
else:
    print('Dependencias OK')


Dependencias OK


In [9]:
print('Este notebook está configurado para uso local: no requiere montar Google Drive.')


Este notebook está configurado para uso local: no requiere montar Google Drive.
